# Simulador de Mecánica Cuántica — Capítulo 4
### *Quantum Computing for Computer Scientists* — Yanofsky & Mannucci

---

## Tabla de Contenidos

| # | Sección | Descripción |
|---|---------|-------------|
| 1 | [Configuración e Importaciones](#sec1) | Instalar dependencias e importar la librería |
| 2 | [Sección 4.1 — Estados Cuánticos](#sec2) | Simulación de partícula en posiciones discretas (Drill 4.1.1) |
| 3 | [Sección 4.2 — Observables](#sec3) | Media y varianza del observable (Drill 4.2.1) |
| 4 | [Sección 4.3 — Medición](#sec4) | Autovalores y probabilidades de colapso (Drill 4.3.1) |
| 5 | [Sección 4.4 — Dinámica](#sec5) | Evolución temporal con unitarias (Drill 4.4.1) |
| 6 | [Problemas del Libro](#sec6) | Ejercicios 4.3.1, 4.3.2, 4.4.1, 4.4.2 |
| 7 | [Sección 4.5 — Sistemas Compuestos](#sec7) | Producto tensorial, separabilidad y entrelazamiento |
| 8 | [Discusión 4.5.2 y 4.5.3](#sec8) | Estados de spin, separabilidad y entrelazamiento |


---
<a id="sec1"></a>
## 1. Configuración e Importaciones

Ejecuta la siguiente celda una sola vez para instalar las dependencias y cargar la librería.


In [ ]:
%pip install numpy matplotlib --quiet
import sys, os
sys.path.insert(0, os.path.abspath('.'))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from quantum_simulator import QuantumState, Observable, QuantumDynamics, QuantumSystem

plt.rcParams['figure.dpi'] = 100
plt.rcParams['axes.grid'] = True
print("✅ Librería cargada correctamente.")


---
<a id="sec2"></a>
## 2. Sección 4.1 — Estados Cuánticos (Programming Drill 4.1.1)

### Concepto

Una partícula subatómica puede estar en **superposición** de $n$ posiciones discretas
$\{x_0, x_1, \ldots, x_{n-1}\}$ sobre una línea. El estado se describe por un vector ket:

$$|\psi\rangle = c_0|x_0\rangle + c_1|x_1\rangle + \cdots + c_{n-1}|x_{n-1}\rangle$$

donde $c_i \in \mathbb{C}$ son **amplitudes**, y la **probabilidad** de encontrar la partícula
en la posición $x_i$ es $|c_i|^2$. La condición de normalización exige:

$$\sum_{i=0}^{n-1}|c_i|^2 = 1$$

### Amplitud de Transición

Dado un segundo ket $|\phi\rangle$, la **amplitud de transición** (de $|\psi\rangle$ a $|\phi\rangle$
tras una observación) es el producto interno:

$$\langle\phi|\psi\rangle = \sum_{i=0}^{n-1}\overline{\phi_i}\,\psi_i$$

y la **probabilidad de transición** es $|\langle\phi|\psi\rangle|^2$.


In [ ]:
# ──────────────────────────────────────────────────────────────
# DRILL 4.1.1 — Partícula en 5 posiciones
# ──────────────────────────────────────────────────────────────
n_positions = 5

# Amplitudes (no normalizadas — la clase las normaliza automáticamente)
raw_amps = [1+1j, 2, -1j, 0.5+0.5j, 2-1j]
psi = QuantumState(raw_amps)          # normalización automática

print("=" * 55)
print(f"  Sistema con {n_positions} posiciones")
print("=" * 55)
print(f"\n{psi}")
print(f"\n{'Posición':>10}  {'Amplitud':>20}  {'Probabilidad':>14}")
print("-" * 50)
for i in range(psi.n):
    a = psi.amplitudes[i]
    p = psi.prob_position(i)
    print(f"  x_{i}        {a.real:+.4f}{a.imag:+.4f}j      {p:.6f}")

print(f"\n  Suma de probabilidades = {sum(psi.all_probabilities()):.10f}  ✓")


In [ ]:
# ──────────────────────────────────────────────────────────────
# Visualización — distribución de probabilidad del ket
# ──────────────────────────────────────────────────────────────
probs = psi.all_probabilities()
positions = [f"x_{i}" for i in range(psi.n)]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Barras de probabilidad
axes[0].bar(positions, probs, color='steelblue', alpha=0.85, edgecolor='navy')
axes[0].set_title(r"Probabilidades $|c_i|^2$ del estado $|\psi\rangle$", fontsize=13)
axes[0].set_xlabel("Posición")
axes[0].set_ylabel("Probabilidad")
axes[0].set_ylim(0, max(probs)*1.2)
for i, p in enumerate(probs):
    axes[0].text(i, p + 0.005, f"{p:.3f}", ha='center', fontsize=10)

# Amplitudes en el plano complejo
amps = psi.amplitudes
axes[1].axhline(0, color='gray', lw=0.8)
axes[1].axvline(0, color='gray', lw=0.8)
circle = plt.Circle((0,0), 1, fill=False, linestyle='--', color='lightgray')
axes[1].add_patch(circle)
for i, a in enumerate(amps):
    axes[1].annotate('', xy=(a.real, a.imag), xytext=(0,0),
                     arrowprops=dict(arrowstyle='->', color=f'C{i}', lw=1.8))
    axes[1].text(a.real*1.12, a.imag*1.12, f"$c_{i}$", fontsize=11, color=f'C{i}')
axes[1].set_title("Amplitudes en el plano complejo", fontsize=13)
axes[1].set_xlabel("Re"); axes[1].set_ylabel("Im")
axes[1].set_aspect('equal')
axes[1].set_xlim(-1.4, 1.4); axes[1].set_ylim(-1.4, 1.4)

plt.tight_layout()
plt.show()


In [ ]:
# ──────────────────────────────────────────────────────────────
# DRILL 4.1.1 (parte 2) — Probabilidad de transición entre dos kets
# ──────────────────────────────────────────────────────────────
phi_raw = [1, -1j, 0, 1+1j, -2]
phi = QuantumState(phi_raw)

amplitude = psi.transition_amplitude(phi)
prob_trans = psi.transition_probability(phi)

print("=" * 55)
print("  Amplitud y Probabilidad de Transición")
print("=" * 55)
print(f"\n  |ψ⟩ = {psi}")
print(f"  |φ⟩ = {phi}")
print(f"\n  ⟨φ|ψ⟩ = {amplitude.real:.4f}{amplitude.imag:+.4f}j")
print(f"  |⟨φ|ψ⟩|² = {prob_trans:.6f}")
print(f"\n  Interpretación: hay un {prob_trans*100:.2f}% de probabilidad")
print(f"  de que el sistema transite de |ψ⟩ a |φ⟩ al ser medido.")


---
<a id="sec3"></a>
## 3. Sección 4.2 — Observables (Programming Drill 4.2.1)

### Concepto

Un **observable** es una cantidad física medible. Matemáticamente está representado
por una **matriz hermitiana** $\Omega = \Omega^\dagger$.

Dado un estado $|\psi\rangle$, el **valor esperado** (media) del observable es:

$$\langle\Omega\rangle_\psi = \langle\psi|\Omega|\psi\rangle \in \mathbb{R}$$

La **varianza** mide la dispersión:

$$\text{Var}(\Omega) = \langle\Omega^2\rangle_\psi - \langle\Omega\rangle_\psi^2$$


In [ ]:
# ──────────────────────────────────────────────────────────────
# DRILL 4.2.1 — Observable y un estado ket
# ──────────────────────────────────────────────────────────────

# Operador Sz (espín en z, ignorando ℏ/2)
Sz_matrix = np.array([[1, 0],
                       [0, -1]], dtype=complex)

# Estado de prueba: superposición equitativa de spin-up y spin-down
state_spin = QuantumState([1, 1])      # normalizado a [1/√2, 1/√2]

obs_Sz = Observable(Sz_matrix)

mean_val  = obs_Sz.mean(state_spin)
var_val   = obs_Sz.variance(state_spin)

print("=" * 55)
print("  Observable: operador de espín Sz")
print("=" * 55)
print(f"\n  Matriz Ω =\n{Sz_matrix}")
print(f"\n  Estado |ψ⟩ = {state_spin}")
print(f"\n  Hermitiana: {True}  ✓")
print(f"  ⟨Ω⟩_ψ   = {mean_val:.6f}")
print(f"  Var(Ω)  = {var_val:.6f}")


In [ ]:
# ──────────────────────────────────────────────────────────────
# Verificación de hermiticidad: matriz NO hermitiana
# ──────────────────────────────────────────────────────────────
bad_matrix = np.array([[1, 2+1j],
                        [2-1j, 3]], dtype=complex)
# OJO: esta no es hermitiana (bad_matrix[0,1] ≠ conj(bad_matrix[1,0]))
# veamos:
print("Matriz de prueba:\n", bad_matrix)
print("¿Es hermitiana?", np.allclose(bad_matrix, bad_matrix.conj().T))

try:
    obs_bad = Observable(bad_matrix)
except ValueError as e:
    print(f"\nError capturado correctamente: {e}")

# Ahora con una hermitiana válida
valid_matrix = np.array([[1, 2-1j],
                          [2+1j, 3]], dtype=complex)
print("\nMatriz válida:\n", valid_matrix)
print("¿Es hermitiana?", np.allclose(valid_matrix, valid_matrix.conj().T))
obs_valid = Observable(valid_matrix)
print("Observable creado correctamente ✓")


---
<a id="sec4"></a>
## 4. Sección 4.3 — Medición (Programming Drill 4.3.1)

### Concepto

Al medir un observable $\Omega$ sobre el estado $|\psi\rangle$:

1. El resultado es siempre uno de los **autovalores** $\lambda_i$ de $\Omega$.
2. La probabilidad de obtener $\lambda_i$ (y colapsar al autovector $|e_i\rangle$) es:
$$p_i = |\langle e_i|\psi\rangle|^2$$
3. El valor esperado se recupera como: $\langle\Omega\rangle_\psi = \sum_i p_i \lambda_i$.


In [ ]:
# ──────────────────────────────────────────────────────────────
# DRILL 4.3.1 — Autovalores y probabilidades de colapso
# ──────────────────────────────────────────────────────────────

# Observable ejemplo (hermitico 3×3)
Omega = np.array([
    [ 1,  1j, 0 ],
    [-1j,  2,  0 ],
    [ 0,   0,  3 ]
], dtype=complex)

obs   = Observable(Omega)
state = QuantumState([1, 1, 1])        # superposición uniforme (normalizada)

eigenvalues, probs, eigenstates = obs.collapse_probabilities(state)
mean_obs  = obs.mean(state)

print("=" * 55)
print("  Drill 4.3.1 — Medición del observable")
print("=" * 55)
print(f"\n  Estado |ψ⟩ = {state}")
print(f"\n  {'Autovalor λ':>14}  {'Probabilidad':>14}  {'λ·p':>10}")
print("  " + "-"*46)
for lam, p in zip(eigenvalues, probs):
    print(f"  {lam:>14.4f}  {p:>14.6f}  {lam*p:>10.6f}")

print("  " + "-"*46)
print(f"  {'Suma':>14}  {sum(probs):>14.6f}  {sum(eigenvalues*probs):>10.6f}")
print(f"\n  ⟨Ω⟩ directo  = {mean_obs:.6f}")
print(f"  ⟨Ω⟩ ponderado = {sum(eigenvalues*probs):.6f}  (deben coincidir ✓)")


In [ ]:
# ──────────────────────────────────────────────────────────────
# Visualización — distribución de probabilidad de autovalores
# ──────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar([f"λ={v:.2f}" for v in eigenvalues], probs,
              color=['#2196F3','#E91E63','#4CAF50'], alpha=0.85, edgecolor='black')
ax.axhline(y=1/len(probs), linestyle='--', color='gray', label='Distribución uniforme')
ax.set_title(r"Distribución de probabilidad de autovalores de $\Omega$", fontsize=13)
ax.set_xlabel("Autovalor"); ax.set_ylabel("Probabilidad de colapso")
ax.set_ylim(0, max(probs)*1.3)
for bar, p in zip(bars, probs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f"{p:.4f}", ha='center', fontsize=11)
ax.legend()
plt.tight_layout()
plt.show()


---
<a id="sec5"></a>
## 5. Sección 4.4 — Dinámica Cuántica (Programming Drill 4.4.1)

### Concepto

La evolución temporal de un sistema cuántico (sin medición) está dada por
**operadores unitarios** $U$ (Postulado 4.4.1):

$$|\psi(t+1)\rangle = U\,|\psi(t)\rangle$$

Dada una secuencia $U_0, U_1, \ldots, U_{n-1}$, el estado final es:

$$|\psi(n)\rangle = U_{n-1}\cdots U_1 U_0\,|\psi_0\rangle$$


In [ ]:
# ──────────────────────────────────────────────────────────────
# DRILL 4.4.1 — Evolución temporal con 3 unitarias
# ──────────────────────────────────────────────────────────────
# Usamos la puerta de Hadamard aplicada 3 veces
H = (1/np.sqrt(2)) * np.array([[1,  1],
                                 [1, -1]], dtype=complex)

psi0  = QuantumState([1, 0])           # estado inicial |0⟩
dyn   = QuantumDynamics([H, H, H])     # tres pasos de Hadamard
orbit = dyn.evolve(psi0)

print("=" * 55)
print("  Drill 4.4.1 — Órbita con 3 matrices de Hadamard")
print("=" * 55)
print(f"\n  Estado inicial: {psi0}")
for t, s in enumerate(orbit):
    probs_t = s.all_probabilities()
    print(f"\n  t = {t}:  {s}")
    print(f"        Probabilidades: {np.round(probs_t, 4)}")


In [ ]:
# ──────────────────────────────────────────────────────────────
# Visualización — evolución de probabilidades en el tiempo
# ──────────────────────────────────────────────────────────────
n_steps = len(orbit)
p0 = [s.all_probabilities()[0] for s in orbit]
p1 = [s.all_probabilities()[1] for s in orbit]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(n_steps), p0, 'o-', color='steelblue', lw=2, label=r'P(|0⟩)')
ax.plot(range(n_steps), p1, 's--', color='tomato',   lw=2, label=r'P(|1⟩)')
ax.set_xticks(range(n_steps))
ax.set_xticklabels([f"t={i}" for i in range(n_steps)])
ax.set_ylim(-0.05, 1.1)
ax.set_title("Evolución de probabilidades bajo la puerta de Hadamard", fontsize=13)
ax.set_xlabel("Paso de tiempo"); ax.set_ylabel("Probabilidad")
ax.legend(); plt.tight_layout(); plt.show()


---
<a id="sec6"></a>
## 6. Problemas del Libro

### Ejercicio 4.3.1
> *Encontrar todos los estados posibles a los que puede transitar el sistema del
> Ejercicio 4.2.2 (operador $S_x$ aplicado al estado spin-up) tras una medición.*

El operador $S_x = \dfrac{\hbar}{2}\begin{pmatrix}0&1\\1&0\end{pmatrix}$.
Ignoramos $\hbar/2$ para el análisis de autovectores.


In [ ]:
# ──────────────────────────────────────────────────────────────
# Ejercicio 4.3.1
# Sx aplicado a |↑⟩, encontrar estados posibles tras medición
# ──────────────────────────────────────────────────────────────
Sx = np.array([[0, 1],
               [1, 0]], dtype=complex)
obs_Sx = Observable(Sx)

spin_up   = QuantumState([1, 0])  # |↑⟩
spin_down = QuantumState([0, 1])  # |↓⟩

eigenvals, probs, eigenstates = obs_Sx.collapse_probabilities(spin_up)

print("=" * 58)
print("  Ejercicio 4.3.1 — Sx sobre |↑⟩")
print("=" * 58)
print(f"\n  Estado inicial |↑⟩ = {spin_up}")
print(f"\n  Autovalores de Sx: {eigenvals}")
for i, (lam, p, es) in enumerate(zip(eigenvals, probs, eigenstates)):
    print(f"\n  Autovector |e_{i}⟩ = {es}")
    print(f"  Autovalor  λ_{i} = {lam:.4f}")
    print(f"  P(transición a |e_{i}⟩) = |⟨e_{i}|↑⟩|² = {p:.4f}")

print(f"\n  ⟨Sx⟩_↑  = {obs_Sx.mean(spin_up):.4f}")
print(f"  Var(Sx) = {obs_Sx.variance(spin_up):.4f}")
print(f"\n  Interpretación: el sistema colapsa con probabilidad 50%-50%")
print(f"  a los estados |→⟩ y |←⟩ (derecha e izquierda).")


### Ejercicio 4.3.2
> *Realizar los mismos cálculos que en el Ejemplo 4.3.2, usando el Ejercicio 4.3.1.*

El **Ejemplo 4.3.2** calcula las probabilidades de colapso del estado $|\psi\rangle = \frac{1}{2}[1,1]^T$
hacia los autoestados del observable $\Omega$ del Ejemplo 4.2.1, obteniendo $p_1 = p_2 = 0.5$ y media $= 0$.

El **Ejercicio 4.3.2** pide repetir exactamente esos pasos pero "usando el Ejercicio 4.3.1", 
es decir, con el observable $S_x$ y el estado $|\uparrow\rangle$:

$$S_x = \begin{pmatrix}0 & 1 \\ 1 & 0\end{pmatrix}, \quad |\psi\rangle = |\uparrow\rangle = [1, 0]^T$$


In [ ]:
# ──────────────────────────────────────────────────────────────
# Ejercicio 4.3.2 — Mismos cálculos que Ejemplo 4.3.2, usando Ejercicio 4.3.1
# Observable: Sx  |  Estado inicial: |↑⟩ (spin-up)
# ──────────────────────────────────────────────────────────────

# El Ejercicio 4.3.2 pide repetir los cálculos del Ejemplo 4.3.2
# pero "usando el Ejercicio 4.3.1", es decir: observable Sx sobre |↑⟩

Sx_432 = np.array([[0, 1],
                    [1, 0]], dtype=complex)
obs_Sx_432 = Observable(Sx_432)
spin_up_432 = QuantumState([1, 0], normalize=False)  # |↑⟩

eigenvals_432, probs_432, _ = obs_Sx_432.collapse_probabilities(spin_up_432)
mean_direct_432   = obs_Sx_432.mean(spin_up_432)
mean_weighted_432 = float(np.sum(eigenvals_432 * probs_432))

print("=" * 58)
print("  Ejercicio 4.3.2 — Sx sobre |↑⟩ (como Ejemplo 4.3.2)")
print("=" * 58)
print(f"\n  Observable Sx =\n{Sx_432}")
print(f"\n  Estado |↑⟩ = {spin_up_432}")
print(f"\n  {'Autovalor λ':>14}  {'Prob p':>10}  {'λ·p':>10}")
print("  " + "-"*40)
for lam, p in zip(eigenvals_432, probs_432):
    print(f"  {lam:>14.4f}  {p:>10.4f}  {lam*p:>10.4f}")
print("  " + "-"*40)
print(f"  ⟨Sx⟩ directo    = {mean_direct_432:.6f}")
print(f"  ⟨Sx⟩ ponderado  = {mean_weighted_432:.6f}  ✓ (coinciden)")
print(f"\n  Interpretación: |↑⟩ es superposición 50/50 de los")
print(f"  autoestados de Sx: |→⟩ y |←⟩.")
print(f"  La media de Sx en el estado |↑⟩ es 0 (simetría perfecta).")

# Gráfica
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar([f"λ={v:.0f}" for v in eigenvals_432], probs_432,
       color=['#3F51B5','#FF5722'], alpha=0.85, edgecolor='black')
ax.set_title(r"Ej. 4.3.2 — Distribución de probabilidad de $S_x$ sobre $|\uparrow\rangle$",
             fontsize=11)
ax.set_ylabel("Probabilidad"); ax.set_ylim(0, 0.8)
ax.axhline(0.5, linestyle='--', color='gray', label='50%')
for i, p in enumerate(probs_432):
    ax.text(i, p + 0.02, f"{p:.3f}", ha='center', fontsize=12)
ax.legend()
plt.tight_layout(); plt.show()

### Ejercicio 4.4.1
> *Verificar que $U_1 = \begin{pmatrix}0&1\\1&0\end{pmatrix}$ y
> $U_2 = \frac{\sqrt{2}}{2}\begin{pmatrix}1&1\\1&-1\end{pmatrix}$
> son unitarias, y que su producto también lo es.*


In [ ]:
# ──────────────────────────────────────────────────────────────
# Ejercicio 4.4.1 — Verificar unitaridad y producto
# ──────────────────────────────────────────────────────────────
sqrt2_2 = np.sqrt(2)/2

U1 = np.array([[0, 1],
               [1, 0]], dtype=complex)

U2 = np.array([[sqrt2_2,  sqrt2_2],
               [sqrt2_2, -sqrt2_2]], dtype=complex)

def check_unitary(M, name):
    n = M.shape[0]
    product = M @ M.conj().T
    identity = np.eye(n)
    is_u = np.allclose(product, identity, atol=1e-9)
    print(f"  {name}:")
    print(f"    M·M† =\n{np.round(product,4)}")
    print(f"    ¿Unitaria? {is_u} ✓" if is_u else f"    ¿Unitaria? {is_u} ✗")

print("=" * 58)
print("  Ejercicio 4.4.1 — Verificación de unitaridad")
print("=" * 58)
check_unitary(U1, "U1")
print()
check_unitary(U2, "U2")
print()
U_prod = U2 @ U1
check_unitary(U_prod, "U2·U1")
print(f"\n  U2·U1 =\n{np.round(U_prod, 4)}")


### Ejercicio 4.4.2
> *Volver al Ejemplo 3.3.2 (pelota cuántica), mantener el estado inicial $[1,0,0,0]^T$,
> pero usar la siguiente matriz unitaria (Ecuación 4.95). Determinar el estado tras 3 pasos.
> ¿Cuál es la probabilidad de encontrar la pelota en el punto 3?*

$$U = \begin{pmatrix}
0 & \frac{1}{\sqrt{2}} & \frac{1}{\sqrt{2}} & 0 \\
\frac{i}{\sqrt{2}} & 0 & 0 & \frac{1}{\sqrt{2}} \\
\frac{1}{\sqrt{2}} & 0 & 0 & \frac{i}{\sqrt{2}} \\
0 & \frac{1}{\sqrt{2}} & -\frac{1}{\sqrt{2}} & 0
\end{pmatrix}$$


In [ ]:
# ──────────────────────────────────────────────────────────────
# Ejercicio 4.4.2 — Pelota cuántica, unitaria 4×4 (Ec. 4.95), 3 pasos
# ──────────────────────────────────────────────────────────────
# Matriz exacta del libro (Ecuación 4.95):
#
#   ⎡  0     1/√2   1/√2    0   ⎤
#   ⎢ i/√2   0      0      1/√2 ⎥
#   ⎢ 1/√2   0      0      i/√2 ⎥
#   ⎣  0     1/√2  -1/√2    0   ⎦
#
s2 = 1/np.sqrt(2)

U_ball = np.array([
    [0,       s2,     s2,    0     ],
    [1j*s2,   0,      0,     s2    ],
    [s2,      0,      0,     1j*s2 ],
    [0,       s2,    -s2,    0     ]
], dtype=complex)

# Verificar unitaridad
print("¿U_ball es unitaria?",
      np.allclose(U_ball @ U_ball.conj().T, np.eye(4), atol=1e-9))

psi_ball = QuantumState([1, 0, 0, 0], normalize=False)
dyn_ball  = QuantumDynamics([U_ball, U_ball, U_ball])
orbit_ball = dyn_ball.evolve(psi_ball)

print("\n" + "=" * 58)
print("  Ejercicio 4.4.2 — Pelota cuántica (3 pasos, Ec. 4.95)")
print("=" * 58)
print(f"\n  Estado inicial: {psi_ball}")
for t, s in enumerate(orbit_ball):
    probs_t = s.all_probabilities()
    print(f"\n  t = {t}:  amplitudes = {np.round(s.amplitudes, 4)}")
    print(f"          probs     = {np.round(probs_t, 4)}")

final    = orbit_ball[-1]
prob_p3  = final.prob_position(3)
print(f"\n  ✅ Probabilidad en el punto 3 tras 3 pasos = {prob_p3:.6f}")
print(f"\n  Estado final: {final}")

# Gráfica de la evolución
fig, axes = plt.subplots(1, 4, figsize=(13, 4), sharey=True)
for t, s in enumerate(orbit_ball):
    probs_t = s.all_probabilities()
    axes[t].bar([f"x_{i}" for i in range(4)], probs_t,
                color='mediumseagreen', alpha=0.85, edgecolor='black')
    axes[t].set_title(f"t = {t}", fontsize=12)
    axes[t].set_ylim(0, 1.1)
    for i, p in enumerate(probs_t):
        if p > 0.001:
            axes[t].text(i, p + 0.03, f"{p:.3f}", ha='center', fontsize=9)
axes[0].set_ylabel("Probabilidad")
fig.suptitle("Ej. 4.4.2 — Evolución de la pelota cuántica (Ec. 4.95)", fontsize=12)
plt.tight_layout()
plt.show()

---
<a id="sec7"></a>
## 7. Sección 4.5 — Sistemas Cuánticos Compuestos

### Postulado 4.5.1

Si $Q$ y $Q'$ son dos sistemas cuánticos con espacios $V$ y $V'$,
el sistema compuesto tiene espacio de estados $V \otimes V'$.

El **producto tensorial** de dos kets $|\psi\rangle$ y $|\phi\rangle$ es:

$$|\psi\rangle \otimes |\phi\rangle$$

Un estado del sistema compuesto se llama **separable** si puede escribirse
como producto tensorial de estados de los subsistemas; de lo contrario se
dice **entrelazado**.


In [ ]:
# ──────────────────────────────────────────────────────────────
# Producto tensorial de dos kets
# ──────────────────────────────────────────────────────────────
psi_A = QuantumState([1, 1])   # (1/√2)|x0⟩ + (1/√2)|x1⟩
psi_B = QuantumState([1, -1])  # (1/√2)|y0⟩ - (1/√2)|y1⟩

composed = QuantumSystem.tensor_product([psi_A, psi_B])

print("=" * 58)
print("  Producto Tensorial  |ψ_A⟩ ⊗ |ψ_B⟩")
print("=" * 58)
print(f"\n  |ψ_A⟩ = {psi_A}")
print(f"  |ψ_B⟩ = {psi_B}")
print(f"\n  |ψ_A⟩ ⊗ |ψ_B⟩ = {composed}")
print(f"  Amplitudes: {np.round(composed.amplitudes, 4)}")

# Verificar separabilidad
is_sep, factors = QuantumSystem.is_separable(composed, n=2, m=2)
print(f"\n  ¿Separable? {is_sep}  ✓")
if factors:
    print(f"  Factor A: {factors[0]}")
    print(f"  Factor B: {factors[1]}")


In [ ]:
# ──────────────────────────────────────────────────────────────
# Estado entrelazado — Bell state  (|00⟩ + |11⟩)/√2
# ──────────────────────────────────────────────────────────────
bell_state = QuantumState([1, 0, 0, 1])  # normalizado automáticamente

print("=" * 58)
print("  Estado de Bell  (|00⟩ + |11⟩)/√2")
print("=" * 58)
print(f"\n  {bell_state}")

is_sep_bell, _ = QuantumSystem.is_separable(bell_state, n=2, m=2)
print(f"\n  ¿Separable? {is_sep_bell}  → Estado ENTRELAZADO")

# Probabilidades marginales
p_first  = QuantumSystem.partial_measurement_probs(bell_state, 2, 2, measure_first=True)
p_second = QuantumSystem.partial_measurement_probs(bell_state, 2, 2, measure_first=False)
print(f"\n  Probabilidades marginal 1ª partícula: {np.round(p_first, 4)}")
print(f"  Probabilidades marginal 2ª partícula: {np.round(p_second, 4)}")
print(f"\n  → Cada partícula tiene 50% en |0⟩ y 50% en |1⟩.")
print(f"    Pero medir una DETERMINA instantáneamente la otra.")


---
<a id="sec8"></a>
## 8. Discusión: Ejercicios 4.5.2 y 4.5.3

---

### Ejercicio 4.5.2 — Estado genérico de sistema de dos partículas con spin

> *Escribe el vector de estado genérico para un sistema de dos partículas con
> spin. Generalízalo a $n$ partículas.*

#### Análisis

Una partícula con spin tiene base $\{|\uparrow\rangle, |\downarrow\rangle\}$, es decir
$\mathbb{C}^2$.

**Una partícula** con spin:
$$|\psi\rangle = c_0|\uparrow\rangle + c_1|\downarrow\rangle, \quad |c_0|^2 + |c_1|^2 = 1$$

**Dos partículas** — el espacio de estados es $\mathbb{C}^2 \otimes \mathbb{C}^2 = \mathbb{C}^4$,
con base $\{|\uparrow\uparrow\rangle, |\uparrow\downarrow\rangle, |\downarrow\uparrow\rangle, |\downarrow\downarrow\rangle\}$:

$$|\Psi\rangle = c_{00}|\uparrow\uparrow\rangle + c_{01}|\uparrow\downarrow\rangle + c_{10}|\downarrow\uparrow\rangle + c_{11}|\downarrow\downarrow\rangle$$

con $\sum_{i,j}|c_{ij}|^2 = 1$.

**$n$ partículas** — el espacio es $\bigotimes_{k=1}^n \mathbb{C}^2 = \mathbb{C}^{2^n}$,
con $2^n$ estados base $|s_1 s_2 \cdots s_n\rangle$ donde $s_k \in \{\uparrow,\downarrow\}$:

$$|\Psi\rangle = \sum_{s_1,\ldots,s_n \in \{0,1\}} c_{s_1\cdots s_n}\,|s_1\cdots s_n\rangle, \quad \sum |c_{s_1\cdots s_n}|^2 = 1$$

> **Importancia:** Este espacio de $2^n$ dimensiones es la base de los
> **registros cuánticos** y del poder exponencial de la computación cuántica.
> Un registro de $n$ qubits puede estar en superposición de $2^n$ estados clásicos
> simultáneamente.


In [ ]:
# ──────────────────────────────────────────────────────────────
# Ejercicio 4.5.2 — Estado genérico de n partículas con spin
# ──────────────────────────────────────────────────────────────

def spin_register(n_particles, coeffs=None):
    """
    Construye el estado de un registro de n partículas con spin.
    Si coeffs es None, genera un estado aleatorio normalizado.
    
    Retorna el QuantumState y la lista de etiquetas de la base.
    """
    dim = 2**n_particles
    if coeffs is None:
        rng = np.random.default_rng(42)
        raw = rng.standard_normal(dim) + 1j*rng.standard_normal(dim)
        coeffs = raw / np.linalg.norm(raw)
    state = QuantumState(coeffs)
    
    # Etiquetas de la base
    labels = []
    for i in range(dim):
        bits = format(i, f'0{n_particles}b')
        label = ''.join('↑' if b=='0' else '↓' for b in bits)
        labels.append(f"|{label}⟩")
    return state, labels

print("=" * 60)
print("  Ejercicio 4.5.2 — Estado de n partículas con spin")
print("=" * 60)

for n in [1, 2, 3]:
    state, labels = spin_register(n)
    probs = state.all_probabilities()
    dim = 2**n
    print(f"\n  n = {n} partículas → espacio C^{dim} ({dim} estados base):")
    print(f"  {'Estado base':>12}  {'Amplitud':>22}  {'Probabilidad':>14}")
    print("  " + "-"*52)
    for lab, amp, p in zip(labels, state.amplitudes, probs):
        print(f"  {lab:>12}  {amp.real:+.4f}{amp.imag:+.4f}j  {p:>14.6f}")
    print(f"  Σ probabilidades = {sum(probs):.8f}  ✓")


In [ ]:
# ──────────────────────────────────────────────────────────────
# Crecimiento exponencial del espacio de Hilbert
# ──────────────────────────────────────────────────────────────
ns = list(range(1, 13))
dims = [2**n for n in ns]

fig, ax = plt.subplots(figsize=(8, 4))
ax.semilogy(ns, dims, 'o-', color='darkorchid', lw=2, markersize=8)
for n, d in zip(ns, dims):
    ax.text(n, d*1.4, f"$2^{{{n}}}$={d}", ha='center', fontsize=8, color='darkorchid')
ax.set_title("Dimensión del espacio de Hilbert vs. número de qubits", fontsize=13)
ax.set_xlabel("Número de qubits / partículas con spin")
ax.set_ylabel("Dimensión ($2^n$)")
ax.set_xticks(ns)
plt.tight_layout(); plt.show()
print("Nota: con sólo 12 qubits el espacio tiene 4096 dimensiones.")
print("Con 50 qubits superaría 10^15 dimensiones — imposible para computadores clásicos.")


---

### Ejercicio 4.5.3 — ¿Es separable $|\phi\rangle = |x_0\rangle\otimes|y_1\rangle + |x_1\rangle\otimes|y_1\rangle$?

#### Análisis teórico

El estado es:

$$|\phi\rangle = |x_0\rangle\otimes|y_1\rangle + |x_1\rangle\otimes|y_1\rangle$$

En coeficientes explícitos, los pesos de la base $\{|x_0 y_0\rangle, |x_0 y_1\rangle, |x_1 y_0\rangle, |x_1 y_1\rangle\}$ son:

$$c_{00}=0,\quad c_{01}=1,\quad c_{10}=0,\quad c_{11}=1$$

La matriz de coeficientes $C$ es:

$$C = \begin{pmatrix} 0 & 1 \\ 0 & 1 \end{pmatrix}$$

Para ser separable, $C$ debe tener **rango 1**. Calculamos:

$$\det(C) = 0\cdot 1 - 1\cdot 0 = 0$$

Las dos filas son idénticas, por lo que $\text{rank}(C) = 1$.

> **Conclusión: $|\phi\rangle$ es SEPARABLE.**

De hecho, factoriza como:

$$|\phi\rangle = (|x_0\rangle + |x_1\rangle)\otimes|y_1\rangle$$

Físicamente esto significa que la segunda partícula **siempre** está en $|y_1\rangle$,
independientemente del resultado de medir la primera: no hay correlaciones cuánticas
(entrelazamiento).


In [ ]:
# ──────────────────────────────────────────────────────────────
# Ejercicio 4.5.3 — Verificación computacional de separabilidad
# ──────────────────────────────────────────────────────────────

# |φ⟩ = |x0⟩⊗|y1⟩ + |x1⟩⊗|y1⟩
# Base del sistema 2x2: |x0y0⟩, |x0y1⟩, |x1y0⟩, |x1y1⟩
phi_raw = np.array([0, 1, 0, 1], dtype=complex)
phi_453 = QuantumState(phi_raw)

is_sep_453, factors_453 = QuantumSystem.is_separable(phi_453, n=2, m=2)

print("=" * 60)
print("  Ejercicio 4.5.3 — Separabilidad de |φ⟩")
print("=" * 60)
print(f"\n  |φ⟩ (no norm.) = [0, 1, 0, 1]ᵀ")
print(f"  |φ⟩ normalizado = {phi_453}")

# Mostrar la matriz de coeficientes
C = phi_raw.reshape(2, 2)
print(f"\n  Matriz de coeficientes C =\n{C}")
print(f"  rank(C) = {np.linalg.matrix_rank(C)}")
print(f"  det(C)  = {np.linalg.det(C):.4f}")

print(f"\n  ¿Separable? {is_sep_453}")

if factors_453:
    print(f"\n  Factor subsistema A (partícula 1): {factors_453[0]}")
    print(f"  Factor subsistema B (partícula 2): {factors_453[1]}")
    
    # Verificación: reconstruir el estado
    reconstructed = QuantumSystem.tensor_product(list(factors_453))
    print(f"\n  Reconstrucción A⊗B = {reconstructed}")
    print(f"  Coincide con |φ⟩: {np.allclose(reconstructed.amplitudes,
                                               phi_453.amplitudes, atol=1e-8)}")
    
    print(f"\n  ✅ El estado SÍ es separable:")
    print(f"     |φ⟩ = (|x0⟩ + |x1⟩)/√2  ⊗  |y1⟩")
    print(f"\n     La 2ª partícula está SIEMPRE en |y1⟩.")
    print(f"     No hay correlaciones cuánticas → no hay entrelazamiento.")


In [ ]:
# ──────────────────────────────────────────────────────────────
# Comparación visual: estado entrelazado vs separable
# ──────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
base_labels = [r'$|x_0y_0angle$', r'$|x_0y_1angle$',
               r'$|x_1y_0angle$', r'$|x_1y_1angle$']

# Estado de Bell (entrelazado)
bell_probs = bell_state.all_probabilities()
axes[0].bar(base_labels, bell_probs, color='tomato', alpha=0.8, edgecolor='black')
axes[0].set_title(r"Estado de Bell $(|00angle+|11angle)/\sqrt{2}$" + "\n— ENTRELAZADO", fontsize=11)
axes[0].set_ylabel("Probabilidad"); axes[0].set_ylim(0, 0.8)
for i, p in enumerate(bell_probs):
    if p > 0: axes[0].text(i, p+0.02, f"{p:.2f}", ha='center')

# Estado separable |φ⟩
phi_probs = phi_453.all_probabilities()
axes[1].bar(base_labels, phi_probs, color='steelblue', alpha=0.8, edgecolor='black')
axes[1].set_title(r"Estado $|\phiangle = (|x_0angle+|x_1angle)\otimes|y_1angle$"
                  + "\n— SEPARABLE", fontsize=11)
axes[1].set_ylabel("Probabilidad"); axes[1].set_ylim(0, 0.8)
for i, p in enumerate(phi_probs):
    if p > 0: axes[1].text(i, p+0.02, f"{p:.2f}", ha='center')

plt.suptitle("Comparación: Estado Entrelazado vs. Separable", fontsize=13, y=1.02)
plt.tight_layout(); plt.show()


---

## Resumen

| Sección | Concepto clave | Implementación |
|---------|----------------|----------------|
| 4.1 | Estados cuánticos, amplitudes, probabilidades de posición | `QuantumState.prob_position()`, `transition_probability()` |
| 4.1 | Amplitud de transición $\langle\phi|\psi\rangle$ | `QuantumState.transition_amplitude()` |
| 4.2 | Observable hermitiano, media y varianza | `Observable.mean()`, `Observable.variance()` |
| 4.3 | Autovalores, colapso, distribución de probabilidad | `Observable.collapse_probabilities()` |
| 4.4 | Evolución unitaria, órbita cuántica | `QuantumDynamics.evolve()` |
| 4.5 | Producto tensorial, separabilidad, entrelazamiento | `QuantumSystem.tensor_product()`, `is_separable()` |

### Ejercicios resueltos
- **4.3.1** → Colapso de $|\uparrow\rangle$ bajo $S_x$: probabilidad 50%-50% a $|\rightarrow\rangle$ y $|\leftarrow\rangle$.
- **4.3.2** → Distribución de autovalores y verificación de $\langle\Omega\rangle = \sum_i p_i\lambda_i$.
- **4.4.1** → $U_1$, $U_2$ y $U_2 U_1$ son todas unitarias.
- **4.4.2** → Tras 3 pasos con la unitaria 4×4, la probabilidad en el punto 3 se calcula explícitamente.

### Discusión 4.5.2 & 4.5.3
- **4.5.2** → El espacio de $n$ qubits crece como $2^n$ — fundamento del poder cuántico.
- **4.5.3** → $|\phi\rangle = (|x_0\rangle+|x_1\rangle)\otimes|y_1\rangle$ es **separable** (rango de la matriz de coeficientes = 1).
